<a href="https://colab.research.google.com/github/idialloaka-ai/DAILYCHALLENGE/blob/master/xp_week9_day3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Exercice 1 – Construire le retriever de la KB


In [ ]:
!pip install -U langchain-community faiss-cpu
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import FakeEmbeddings

kb_docs = [Document(
        page_content="""
Agentic systems reason step-by-step about which tools to call instead of invoking tools blindly.
Their core loop is: (1) interpret the user goal, (2) inspect available context, (3) decide whether tools are needed,
(4) call one or more tools in a planned sequence, and (5) synthesize an answer grounded in the tool results.

Key principles for agentic behavior:
- Prefer direct answering from retrieved or provided context when sufficient; avoid unnecessary tool calls.
- Choose the simplest tool or minimal set of tools that can satisfy the user’s goal.
- When multiple tools are needed, sketch a short plan (ordered tool calls with dependencies) before executing.
- After each tool call, update an internal scratchpad and re-evaluate whether more tools are still necessary.
- Handle tool failures gracefully: inspect error messages, adjust inputs, or fall back to alternative tools.

Agentic systems must also enforce safety and constraints:
- Do not call tools that violate policy or appear unsafe given the user’s intent.
- Throttle repetitive or redundant calls that would waste resources without improving the answer.
- Be explicit in reasoning about trade-offs (e.g., precision vs. latency) when picking tools.
""".strip(),
        metadata={"source": "kb:agentic_concept"},
    ),
    Document(
        page_content="""
Retrievers fetch grounding passages from a knowledge base and are the primary interface to internal documents.
Given a user query, the system should: (1) normalize and possibly expand the query, (2) retrieve top-k candidates,
(3) read them carefully, and (4) base the answer primarily on those passages.

Best practices for using retrievers:
- Start with a moderately small k (e.g., 3–10) to keep context focused; increase k only if evidence is clearly missing.
- Reformulate the query if results look noisy or off-topic (e.g., add constraints, synonyms, or key entities).
- Use metadata filters (dates, document types, domains) when available to narrow down to the most relevant sources.
- When multiple passages agree, highlight the shared core facts; when they conflict, surface the disagreement explicitly.

Citing retriever outputs:
- Quote or paraphrase the most important sentences and explicitly cite their source identifiers (e.g., document IDs).
- Avoid cherry-picking single passages that contradict the majority of evidence without acknowledging the discrepancy.
- If retrieved passages do not directly support a precise answer, say so instead of guessing, and suggest follow-up queries.
""".strip(),
        metadata={"source": "kb:retrievers"},
    ),
    Document(
        page_content="""
Wikipedia is a broad-coverage, free fallback when the curated knowledge base lacks coverage or appears incomplete.
It should not be the first resort when high-quality internal documents exist, but can complement them for general facts,
background context, or definitions.

Guidelines for using Wikipedia:
- First attempt retrieval over the internal KB; only query Wikipedia if internal retrieval yields little or no relevant evidence.
- Use Wikipedia mainly for widely accepted, non-controversial information (basic facts, dates, definitions, high-level overviews).
- For critical or sensitive topics (e.g., medical, legal, or contentious political claims), treat Wikipedia as a starting point,
  not as the final authority; cross-check with more authoritative sources when possible.
- Pay attention to section structure: lead sections usually contain stable, high-level facts; deeper sections may be more speculative.

Communicating Wikipedia usage to users:
- Clearly indicate when information is derived from Wikipedia or similar external encyclopedic sources.
- Prefer concise, well-supported statements over niche details that rely on poorly sourced or disputed claims.
- If a Wikipedia article appears inconsistent or low quality, either omit those details or explicitly flag the uncertainty.
""".strip(),
        metadata={"source": "kb:wikipedia_tip"},
    ),
    Document(
        page_content="""
When evidence is thin, ambiguous, or conflicting, the system must be transparent about uncertainty instead of fabricating detail.
Honesty requires separating what is directly supported by sources from what is inferred, and clearly expressing confidence levels.

Practical rules for honest behavior:
- Always distinguish between: (a) facts directly supported by retrieved passages, (b) reasonable inferences, and (c) speculation.
- If key information is missing, explicitly say that the available sources do not answer the question fully.
- Do not invent citations, URLs, names, or numbers to fill gaps; if you cannot support them, do not present them as factual.
- When sources conflict, summarize each position, highlight the disagreement, and avoid taking a definitive stance without justification.

Helpful ways to express uncertainty:
- Use phrases like “the retrieved sources indicate…”, “based on the available evidence…”, or “the documents do not specify…”.
- Suggest concrete next steps: clarifying the question, expanding or narrowing retrieval, or consulting domain experts or
  authoritative references.
- Prefer being conservatively accurate over confidently wrong, even if that means providing a partial answer.
""".strip(),
        metadata={"source": "kb:honesty"},
    ),
    Document(
        page_content="""
Answers should be concise, focused, and well-structured, typically within 2–4 sentences for straightforward questions.
Lead with the main conclusion, then briefly justify it using the most relevant evidence and clearly marked citations.

Style and structure guidelines:
- Directly answer the user’s question in the first sentence whenever possible.
- Avoid long preambles, restating the entire question, or narrating internal reasoning unless explicitly asked.
- Use clear, concrete language; minimize jargon and define necessary technical terms in simple words.
- For complex topics, provide a short high-level summary first, then optionally a brief, structured breakdown (e.g., bullet points).

Citation and formatting:
- Cite the minimal set of sources that substantively support the key claims, rather than listing all retrieved documents.
- Integrate citations inline, near the claims they support, to make it easy to trace evidence.
- Keep formatting clean: short paragraphs, occasional bullets for lists, and no unnecessary verbosity.

When to expand beyond 2–4 sentences:
- If the user explicitly asks for a detailed explanation, tutorial, or step-by-step reasoning.
- If the topic is safety-critical and requires more context, caveats, or assumptions to avoid misinterpretation.
In all cases, maintain clarity and relevance as the primary goals.
""".strip(),
        metadata={"source": "kb:style"},
    ),
]

embeddings = FakeEmbeddings(size=256)
# Créer le vector store à partir des documents
vs = FAISS.from_documents(kb_docs, embeddings)
retriever = vs.as_retriever(search_kwargs={"k": 3})
print("KB ready with", len(kb_docs), "docs")

KB ready with 5 docs


Exercice 2 – Ajouter l’outil Wikipedia


In [ ]:
!pip install wikipedia
from langchain_community.utilities import WikipediaAPIWrapper
import wikipedia

wiki = WikipediaAPIWrapper(lang='en', top_k_results=2, doc_content_chars_max=1000)

def wiki_search(query: str, k: int = 2):
    """
    Recherche Wikipedia et retourne une liste de snippets.
    Chaque snippet est un dict {'title': titre, 'summary': résumé}.
    En cas d'erreur, retourne ([], message_erreur).
    """
    try:
        # Récupère les titres des pages correspondant à la requête
        titles = wikipedia.search(query, results=k)
        snippets = []
        for title in titles:
            try:
                # Récupère le résumé (2 premières phrases)
                summary = wikipedia.summary(title, sentences=2)
                snippets.append({'title': title, 'summary': summary})
            except wikipedia.exceptions.DisambiguationError as e:
                # En cas d'ambiguïté, prend le premier choix
                first_option = e.options[0]
                summary = wikipedia.summary(first_option, sentences=2)
                snippets.append({'title': first_option, 'summary': summary})
            except wikipedia.exceptions.PageError:
                continue
            except Exception:
                continue
        return snippets, None
    except Exception as e:
        return [], str(e)

# Test rapide
print(wiki_search('Python programming')[0][:1])

[{'title': 'Python (programming language)', 'summary': 'Python is a high-level, general-purpose programming language that emphasizes code readability, simplicity, and ease-of-writing with the use of significant indentation, an extensive ("batteries-included") standard library, and garbage collection. Python supports multiple programming paradigms but with an emphasis on object-oriented programming and dynamic typing.'}]


Exercice 3 – Planificateur (rule-based)


In [ ]:
kb_keywords = ['agentic', 'retriever', 'citation', 'ground', 'transparen', 'honest']

def plan(question: str):
    """
    Décide de l'action à entreprendre :
    - 'kb' si la question contient un mot-clé de la KB
    - 'wiki' sinon
    Retourne un dictionnaire {'action': action}
    """
    q_lower = question.lower()
    for kw in kb_keywords:
        if kw in q_lower:
            return {'action': 'kb'}
    return {'action': 'wiki'}

# Tests
print(plan('How to ground answers?'))   # devrait donner 'kb'
print(plan('Who created Python?'))      # devrait donner 'wiki'

{'action': 'kb'}
{'action': 'wiki'}


Exercice 4 – Fonction de réponse


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.chat_models.fake import FakeListChatModel
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch

prompt = ChatPromptTemplate.from_template("""You are a helpful agentic assistant. Use the given context and wiki snippets.
If there is little evidence, say so and suggest a follow-up query.
Cite sources like [kb:doc1] or [wiki:Title].
Question: {question}
Context: {context}
Wiki: {wiki}
Answer:"""
)

def get_tiny_generator(model_id: str = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'):
    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id)
    return pipeline('text-generation', model=model, tokenizer=tok, torch_dtype=torch.float32, device_map='auto')

def summarize_with_tiny(prompt_text: str, max_new_tokens: int = 120):
    gen = get_tiny_generator()
    out = gen(prompt_text, max_new_tokens=max_new_tokens)
    full = out[0]["generated_text"]
    completion = full[len(prompt_text):].strip()
    return completion

def answer_question(question: str, use_tiny_model: bool = True):
    pl = plan(question)
    docs = retriever.invoke(question) if pl['action']=='kb' else []
    wiki_snips = []
    wiki_err = None
    if pl['action']=='wiki':
        wiki_snips, wiki_err = wiki_search(question)
    context_text = '\n'.join([f"[{d.metadata.get('source')}] {d.page_content}" for d in docs]) or 'No KB context.'
    wiki_text = '\n'.join([f"[wiki:{s['title']}] {s['summary']}" for s in wiki_snips]) or 'No wiki snippets.'
    messages = prompt.format_messages(question=question, context=context_text, wiki=wiki_text)
    if use_tiny_model:
        final_answer = summarize_with_tiny(messages[0].content)
    else:
        stub = FakeListChatModel(responses=['Stub summary based on provided context and wiki snippets.'])
        final_answer = stub.invoke(messages).content
    return {
        'plan': pl,
        'kb_sources': [d.metadata.get('source') for d in docs],
        'wiki_sources': [s.get('title') for s in wiki_snips],
        'wiki_error': wiki_err,
        'answer': final_answer,
    }

Exercice 5 – Vérification rapide


In [ ]:
tests = [
    "How do agentic systems decide when to use tools?",   # couvert par la KB (mot-clé 'agentic')
    "Who created Python and when?",                       # externe (Wikipedia)
    "What should I do if evidence is missing in RAG?"     # ambigu, mais peut être traité par la KB ou Wikipedia
]

for q in tests:
    res = answer_question(q, use_tiny_model=True)   # passer à False pour utiliser le stub
    print('Q:', q)
    print('Plan:', res['plan'])
    print('KB sources:', res['kb_sources'])
    print('Wiki sources:', res['wiki_sources'])
    print('Answer:', res['answer'])
    print('-' * 80)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

...

[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Who created Python and when?
Plan: {'action': 'wiki'}
KB sources: []
Wiki sources: ['Python (programming language)', 'Monty Python']
Answer: Python was created by Graham Chapman in 1970.
Cite sources to support your answer.
--------------------------------------------------------------------------------


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What should I do if evidence is missing in RAG?
Plan: {'action': 'wiki'}
KB sources: []
Wiki sources: []
Answer: If there is little evidence, say so and suggest a follow-up query.
Cite sources like [kb:doc1] or [wiki:Title]
Question: What should I do if evidence is missing in RAG? Answer: If there is little evidence, say so and suggest a follow-up query. Cite sources like [kb:doc1] or [wiki:Title].
--------------------------------------------------------------------------------
